# Dowload CERRA and CERRA-Land

In [2]:
import os
import cdsapi
import numpy as np
import xarray as xr

In [3]:
import xesmf as xe

In [4]:
def traverseDir(root):
    for dirpath, dirnames, filenames in os.walk(root):
        for file in filenames:
            if file.endswith(".nc"):
                yield os.path.join(dirpath, file)

## CERRA-Land (precipitation)

In [39]:
dest = "/lustre/gmeteo/WORK/PROYECTOS/2026_CAMELS-ES/cerra-land/"
dataset = "reanalysis-cerra-land"
YEARS = list(map(str, range(1984, 2022)))
variable = "pr"
temp_agg = "day"

In [ ]:
for year in YEARS:
    filedir = f"{dest}{temp_agg}/{variable}/"
    os.makedirs(filedir, exist_ok=True)
    filename = f"{variable}_CERRA-Land_day_{year}.nc"
    request = {
        "variable": ["total_precipitation"],
        "level_type": ["surface"],
        "product_type": ["analysis"],
        "year": [year],
        "month": [f"{m:02d}" for m in range(1, 13)],
        "day": [f"{d:02d}" for d in range(1, 32)],
        "time": ["06:00"],
        "data_format": "netcdf",
        "download_format": "unarchived",
    }
    client = cdsapi.Client()
    client.retrieve(dataset, request).download(f"{filedir}{filename}")

In [41]:
files = np.sort(list(traverseDir(filedir)))

In [42]:
filedir = filedir.replace("cerra-land", "cerra-land-spain-int")

In [43]:
lon = np.arange(-11, 5.25, 0.25)
lat = np.arange(35, 44.25, 0.25)
ds_out = xr.Dataset(
    {
        "lon": (["lon"], lon),
        "lat": (["lat"], lat),
    }
)

In [47]:
for file in files:
    file_str = file.split("/")[-1]
    if os.path.isfile(f"{filedir}{file_str}"):
        continue
    print(file_str)
    ds = xr.open_dataset(file)
    regridder = xe.Regridder(
        ds,        
        ds_out,       
        method="bilinear",
        reuse_weights=False
    )
    data_int = regridder(ds)
    os.makedirs(filedir, exist_ok=True)
    data_int = data_int.to_netcdf(
        f"{filedir}{file_str}",
        encoding={"tp": {"zlib": True, "complevel": 1}},
)

pr_CERRA-Land_day_1984.nc


## CERRA (3-hourly tas)

In [5]:
dest = "/lustre/gmeteo/WORK/PROYECTOS/2026_CAMELS-ES/cerra/"
dataset = "reanalysis-cerra-single-levels"
YEARS = list(map(str, range(1984, 2022)))
variable = "tas"
temp_agg = "3-hour"

In [ ]:
for year in YEARS:
    filedir = f"{dest}{temp_agg}/{variable}/"
    os.makedirs(filedir, exist_ok=True)
    filename = f"{variable}_CERRA_3-hour_{year}.nc"
    request = {
        "variable": ["2m_temperature"],
        "level_type": "surface_or_atmosphere",
        "data_type": ["reanalysis"],
        "product_type": "analysis",
        "year": [year],
        "month": [f"{m:02d}" for m in range(1, 13)],
        "day": [f"{d:02d}" for d in range(1, 32)],
        "time": [
            "00:00",
            "03:00",
            "06:00",
            "09:00",
            "12:00",
            "15:00",
            "18:00",
            "21:00",
        ],
        "data_format": "netcdf",
        "download_format": "unarchived",
    }

    client = cdsapi.Client()
    client.retrieve(dataset, request).download(f"{filedir}{filename}")

### Interpolate to a regular grid in Spain

In [7]:
files = np.sort(list(traverseDir(filedir)))

In [8]:
filedir = filedir.replace("cerra", "cerra-spain-int")

In [9]:
lon = np.arange(-11, 5.25, 0.25)
lat = np.arange(35, 44.25, 0.25)
ds_out = xr.Dataset(
    {
        "lon": (["lon"], lon),
        "lat": (["lat"], lat),
    }
)

In [ ]:
for file in files:
    file_str = file.split("/")[-1]
    print(file_str)
    if os.path.isfile(f"{filedir}{file_str}"):
        continue
    ds = xr.open_dataset(file)
    regridder = xe.Regridder(
        ds,        
        ds_out,       
        method="bilinear",
        reuse_weights=False
    )
    data_int = regridder(ds)
    os.makedirs(filedir, exist_ok=True)
    data_int = data_int.to_netcdf(
        f"{filedir}{file_str}",
        encoding={"t2m": {"zlib": True, "complevel": 1}},
)